## 1 - Project Setup
This section initializes the workspace, mounts Google Drive (on Colab) or resolves local VM
paths, extracts the dataset, and authenticates with HuggingFace Hub.

**Note:** DINOv3 checkpoints on HuggingFace are gated. You need to accept the license for both
https://huggingface.co/facebook/dinov3-convnext-small-pretrain-lvd1689m and
https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m and have an access token with
read permission before running the login cell below.

In [ ]:
%pip install -q transformers huggingface_hub accelerate

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import time
import torch.nn as nn
from huggingface_hub import login
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from transformers import AutoModel
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_curve, auc
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Environment setup: works both on Colab (Drive-mounted) and on a local VM/JupyterLab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DRIVE_ROOT = '/content/drive/MyDrive/internship-deepfake-forensic'
    DATASET_DIR = '/content/dataset'
    MODELS_DIR = os.path.join(DRIVE_ROOT, 'deepfake_models_final')

    if not os.path.exists(DATASET_DIR):
        print("Extracting the dataset... (this might take a few minutes)")
        !unzip -q {os.path.join(DRIVE_ROOT, 'deepfake_dataset.zip')} -d {DATASET_DIR}
        print("Extraction completed!")
    else:
        print("Dataset already present and ready to use!")
else:
    DATA_ROOT = '/home/lucia_pola/internship-deepfake-forensic/deepfake-forensics-pipeline/02_Extended_Framework'
    DATASET_DIR = os.path.join(DATA_ROOT, 'Datasets')
    MODELS_DIR = os.path.join(DATA_ROOT, 'deepfake_models_final')
    print(f"Running outside Colab. Using local paths under: {DATA_ROOT}")

os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Dataset directory: {DATASET_DIR}")
print(f"Models directory:  {MODELS_DIR}")

In [ ]:
# Log in to HuggingFace Hub to access the gated DINOv3 checkpoints
login()

## 2 - Dataset and Models Architecture (Hybrid+DINO vs CNN+DINO)

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, real_dirs, fake_dirs, transform=None):
        self.filepaths, self.labels = [], []
        self.transform = transform
        extensions = ('*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG')
        for d in real_dirs:
            for ext in extensions:
                paths = sorted(glob.glob(os.path.join(d, ext)))
                self.filepaths.extend(paths)
                self.labels.extend([0] * len(paths))
        for d in fake_dirs:
            for ext in extensions:
                paths = sorted(glob.glob(os.path.join(d, ext)))
                self.filepaths.extend(paths)
                self.labels.extend([1] * len(paths))

    def __len__(self): return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor([self.labels[idx]], dtype=torch.float32)


test_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # Fail-safe resize
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
class CrossAttentionBlock(nn.Module):
    """
    One LSFA-style fusion round: query = DINOv3 tokens, key/value = CNN tokens.
    """
    def __init__(self, dim, num_heads=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)
        self.cross_attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_ffn = nn.LayerNorm(dim)
        hidden_dim = int(dim * mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim), nn.Dropout(dropout)
        )

    def forward(self, query_tokens, kv_tokens):
        q = self.norm_q(query_tokens)
        kv = self.norm_kv(kv_tokens)
        attn_out, _ = self.cross_attn(q, kv, kv)
        query_tokens = query_tokens + attn_out
        query_tokens = query_tokens + self.ffn(self.norm_ffn(query_tokens))
        return query_tokens

class HybridDinoViTClassifier(nn.Module):
    """
    Hybrid EfficientNet-B0 (trainable, self-attention gated) + DINOv3 ViT-Small (frozen),
    fused via a stack of cross-attention blocks. See HybridDINOv3ViT/step1_baseline.ipynb
    for the full design rationale (the thesis' proposed final architecture).
    """
    def __init__(self, num_classes=1, dino_model_id="facebook/dinov3-vits16-pretrain-lvd1689m",
                 num_fusion_blocks=3, num_heads=4):
        super().__init__()
        self.cnn_backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT).features
        cnn_channels = 1280
        self.cnn_attention = nn.Sequential(
            nn.Conv2d(cnn_channels, cnn_channels // 8, 1, bias=False),
            nn.BatchNorm2d(cnn_channels // 8), nn.ReLU(inplace=True),
            nn.Conv2d(cnn_channels // 8, cnn_channels, 1, bias=False), nn.Sigmoid()
        )

        self.dino_backbone = AutoModel.from_pretrained(dino_model_id)
        for p in self.dino_backbone.parameters():
            p.requires_grad = False
        dino_dim = self.dino_backbone.config.hidden_size

        self.cnn_proj = nn.Linear(cnn_channels, dino_dim)

        self.fusion_blocks = nn.ModuleList([
            CrossAttentionBlock(dino_dim, num_heads=num_heads) for _ in range(num_fusion_blocks)
        ])

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(dino_dim, 256), nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(256, num_classes)
        )

    def train(self, mode=True):
        super().train(mode)
        self.dino_backbone.eval()
        return self

    def state_dict(self, *args, **kwargs):
        full = super().state_dict(*args, **kwargs)
        return {k: v for k, v in full.items() if not k.startswith('dino_backbone.')}

    def load_state_dict(self, state_dict, *args, **kwargs):
        return super().load_state_dict(state_dict, *args, strict=False, **kwargs)

    def forward(self, x):
        cnn_features = self.cnn_backbone(x)
        cnn_features = cnn_features * self.cnn_attention(cnn_features)
        B, C, H, W = cnn_features.shape
        cnn_tokens = cnn_features.flatten(2).transpose(1, 2)
        cnn_tokens = self.cnn_proj(cnn_tokens)

        with torch.no_grad():
            dino_out = self.dino_backbone(pixel_values=x)
            dino_tokens = getattr(dino_out, "last_hidden_state", None)
            if dino_tokens is None:
                dino_tokens = dino_out[0]

        fused = dino_tokens
        for block in self.fusion_blocks:
            fused = block(fused, cnn_tokens)

        pooled = fused.mean(dim=1)
        return self.classifier(pooled)

class DualBranchDinoViTClassifier(nn.Module):
    """
    CNN + DINOv3 ViT-Small: plain EfficientNet-B0 (no self-attention gating) + DINOv3
    ViT-Small (frozen), fused via the same cross-attention stack as HybridDinoViTClassifier.
    See HybridDINOv3ViT/step1_baseline.ipynb for the full design rationale.
    """
    def __init__(self, num_classes=1, dino_model_id="facebook/dinov3-vits16-pretrain-lvd1689m",
                 num_fusion_blocks=3, num_heads=4):
        super().__init__()

        # --- CNN branch: plain EfficientNet-B0, no attention gating (trainable) ---
        self.cnn_backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT).features
        cnn_channels = 1280

        # --- DINOv3 ViT-Small branch (frozen foundation model) ---
        self.dino_backbone = AutoModel.from_pretrained(dino_model_id)
        for p in self.dino_backbone.parameters():
            p.requires_grad = False
        dino_dim = self.dino_backbone.config.hidden_size

        # Project CNN tokens into DINOv3's embedding space for cross-attention
        self.cnn_proj = nn.Linear(cnn_channels, dino_dim)

        # --- Cross-attention fusion stack (trainable) -- identical to the Proposta Finale ---
        self.fusion_blocks = nn.ModuleList([
            CrossAttentionBlock(dino_dim, num_heads=num_heads) for _ in range(num_fusion_blocks)
        ])

        # --- Classification head ---
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(dino_dim, 256), nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(256, num_classes)
        )

    def train(self, mode=True):
        super().train(mode)
        self.dino_backbone.eval()
        return self

    def state_dict(self, *args, **kwargs):
        full = super().state_dict(*args, **kwargs)
        return {k: v for k, v in full.items() if not k.startswith('dino_backbone.')}

    def load_state_dict(self, state_dict, *args, **kwargs):
        return super().load_state_dict(state_dict, *args, strict=False, **kwargs)

    def forward(self, x):
        # CNN branch (trainable, no gating)
        cnn_features = self.cnn_backbone(x)                        # [B, 1280, H, W]
        B, C, H, W = cnn_features.shape
        cnn_tokens = cnn_features.flatten(2).transpose(1, 2)       # [B, H*W, 1280]
        cnn_tokens = self.cnn_proj(cnn_tokens)                     # [B, H*W, dino_dim]

        # DINOv3 branch (frozen)
        with torch.no_grad():
            dino_out = self.dino_backbone(pixel_values=x)
            dino_tokens = getattr(dino_out, "last_hidden_state", None)
            if dino_tokens is None:
                dino_tokens = dino_out[0]

        # Cross-attention fusion: DINOv3 tokens (query) attend to CNN tokens (key/value)
        fused = dino_tokens
        for block in self.fusion_blocks:
            fused = block(fused, cnn_tokens)

        pooled = fused.mean(dim=1)
        return self.classifier(pooled)

## 3 - Data Configuration

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {device}")

path_baseline_hybriddino  = os.path.join(MODELS_DIR, 'baseline_hybriddino', 'path')
path_retrain_hybriddino   = os.path.join(MODELS_DIR, 'retrain_hybriddino', 'path')
path_baseline_dualdino    = os.path.join(MODELS_DIR, 'baseline_dualdino', 'path')
path_retrain_dualdino     = os.path.join(MODELS_DIR, 'retrain_dualdino', 'path')
BATCH = 32
NUM_WORKERS = 4

fei_real_dir, fei_fake_dir = f'{DATASET_DIR}/FEI_MORPHV2_DATASET/test/original/', f'{DATASET_DIR}/FEI_MORPHV2_DATASET/test/fake/'
celeb_real_dir, celeb_fake_dir = f'{DATASET_DIR}/CELEBDFV3_DATASET/test/original/', f'{DATASET_DIR}/CELEBDFV3_DATASET/test/fake/'

ts_reals = [fei_real_dir, celeb_real_dir]
ts_fakes = [fei_fake_dir, celeb_fake_dir]

# Test Dataloader (merged, both sources together -- used by Section 6, Intra-Dataset Evaluation)
test_loader = DataLoader(DeepfakeDataset(ts_reals, ts_fakes, test_transforms), BATCH, shuffle=False, num_workers=NUM_WORKERS)

# Per-source test loaders (FEI only / Celeb-DF++ only -- used by Section 8, to check whether
# the merged metrics above mask a gap between how well each source is recognized)
test_loader_fei = DataLoader(DeepfakeDataset([fei_real_dir], [fei_fake_dir], test_transforms), BATCH, shuffle=False, num_workers=NUM_WORKERS)
test_loader_celeb = DataLoader(DeepfakeDataset([celeb_real_dir], [celeb_fake_dir], test_transforms), BATCH, shuffle=False, num_workers=NUM_WORKERS)

print(f"FEI test samples:   {len(test_loader_fei.dataset)}")
print(f"Celeb test samples: {len(test_loader_celeb.dataset)}")
print(f"Merged test samples: {len(test_loader.dataset)}")

## 4 - Load Models

In [ ]:
print("Loading Models for Direct Comparison...")

#Hybrid EfficientNet-B0 + DINOv3 ViT-Small (Proposta Finale, cross-attention fusion)
model_hybriddino_base = HybridDinoViTClassifier().to(device)
model_hybriddino_base.load_state_dict(torch.load(os.path.join(path_baseline_hybriddino, "hybrid_dinovit_step1.pth"), map_location=device))

model_hybriddino_adv = HybridDinoViTClassifier().to(device)
model_hybriddino_adv.load_state_dict(torch.load(os.path.join(path_retrain_hybriddino, "hybrid_dinovit_step2.pth"), map_location=device))

#EfficientNet-B0 (plain) + DINOv3 ViT-Small (CNN + DINO, same cross-attention fusion as above)
model_dualdino_base = DualBranchDinoViTClassifier().to(device)
model_dualdino_base.load_state_dict(torch.load(os.path.join(path_baseline_dualdino, "dual_dinovit_step1.pth"), map_location=device))

model_dualdino_adv = DualBranchDinoViTClassifier().to(device)
model_dualdino_adv.load_state_dict(torch.load(os.path.join(path_retrain_dualdino, "dual_dinovit_step2.pth"), map_location=device))

trained_models = {
    "Hybrid+DINOv3 ViT-S (Step 1)": model_hybriddino_base,
    "Hybrid+DINOv3 ViT-S (Step 2)": model_hybriddino_adv,
    "CNN+DINOv3 ViT-S (Step 1)": model_dualdino_base,
    "CNN+DINOv3 ViT-S (Step 2)": model_dualdino_adv,
}

print("Models loaded and ready for the showdown!")

## 5 - Test Engine

In [ ]:
def get_all_predictions(model, loader, device):
    model.eval()
    all_labels, all_probs, all_preds = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images).view(-1)

            probs = torch.sigmoid(outputs)
            preds = (outputs > 0.0).float()

            all_labels.extend(labels.view(-1).cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    return np.array(all_labels), np.array(all_probs), np.array(all_preds)


def plot_final_evaluation(models_dict, loader, device, phase_name):
    print(f"\nStarting Quantitative Test for: {phase_name}")

    num_models = len(models_dict)
    fig, axes = plt.subplots(1, num_models + 1, figsize=(6 * (num_models + 1), 6))
    fig.suptitle(f"Quantitative Results - {phase_name}", fontsize=20, fontweight='bold')

    ax_roc = axes[-1]
    ax_roc.plot([0, 1], [0, 1], 'k--', label='Random')

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

    for i, (name, model) in enumerate(models_dict.items()):
        y_true, y_probs, y_preds = get_all_predictions(model, loader, device)

        # Metrics
        acc = accuracy_score(y_true, y_preds)
        f1 = f1_score(y_true, y_preds)
        prec = precision_score(y_true, y_preds)
        rec = recall_score(y_true, y_preds)

        print(f"[{name}] Acc: {acc:.4f} | F1: {f1:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f}")

        # Confusion Matrix
        cm = confusion_matrix(y_true, y_preds)
        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            ax=axes[i],
            cbar=False,
            xticklabels=['REAL', 'FAKE'],
            yticklabels=['REAL', 'FAKE']
        )

        axes[i].set_title(f"{name}")
        if i == 0:
            axes[i].set_ylabel('Ground Truth')
        axes[i].set_xlabel('Predicted')

        # ROC Curve
        fpr, tpr, _ = roc_curve(y_true, y_probs)
        roc_auc = auc(fpr, tpr)

        ax_roc.plot(
            fpr,
            tpr,
            color=colors[i % len(colors)],
            lw=2.5,
            label=f"{name} (AUC: {roc_auc:.4f})"
        )

    ax_roc.set_title("ROC Curves")
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate")
    ax_roc.legend(loc="lower right", fontsize=9)

    plt.tight_layout()
    plt.show()

## 6 - Experimental Results

### 6.1 - Intra-Dataset Evaluation

In [ ]:
plot_final_evaluation(trained_models, test_loader, device, "Hybrid+DINOv3 ViT-S vs CNN+DINOv3 ViT-S")

## 8 - FEI vs Celeb-DF++ Disaggregated Evaluation
The merged test set combines two datasets that are structurally very different (FEI: identity
morphing between two subjects; Celeb-DF++: video face-swap/reenactment/talking-face across 22
methods), and are represented in very different proportions inside the merged test set. A model
could reach a good *merged* accuracy while performing poorly on one of the two, if the source
that is easier to classify dominates the sample count. This section reruns the same metrics as
Section 6, but on `test_loader_fei` and `test_loader_celeb` separately (defined in Section 3),
to check whether that is happening.

In [ ]:
def evaluate_by_source(models_dict, loader, device, source_name):
    rows = []
    for name, model in models_dict.items():
        y_true, y_probs, y_preds = get_all_predictions(model, loader, device)
        rows.append({
            "Model": name,
            "Source": source_name,
            "N": len(y_true),
            "Acc": accuracy_score(y_true, y_preds),
            "F1": f1_score(y_true, y_preds),
            "Prec": precision_score(y_true, y_preds),
            "Rec": recall_score(y_true, y_preds),
            "AUC": auc(*roc_curve(y_true, y_probs)[:2]),
        })
    return pd.DataFrame(rows)


print("Evaluating on FEI-only test samples...")
results_fei = evaluate_by_source(trained_models, test_loader_fei, device, "FEI")

print("Evaluating on Celeb-DF++-only test samples...")
results_celeb = evaluate_by_source(trained_models, test_loader_celeb, device, "Celeb-DF++")

results_by_source = pd.concat([results_fei, results_celeb], ignore_index=True)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
display(results_by_source.sort_values(["Model", "Source"]).reset_index(drop=True))

# Side-by-side view: one row per model, FEI and Celeb-DF++ next to each other, for EVERY
# metric of Tabella 2 (Acc, F1, Prec, Rec, AUC), plus the gap between them -- a large gap
# on any metric means the merged metric in Section 6 is hiding a real difference in how
# well each source is recognized.
metrics = ["Acc", "F1", "Prec", "Rec", "AUC"]
pivots = {}
for metric in metrics:
    piv = results_by_source.pivot(index="Model", columns="Source", values=metric)
    piv["Gap (FEI - Celeb-DF++)"] = piv["FEI"] - piv["Celeb-DF++"]
    pivots[metric] = piv
    print(f"\n{metric} by source (positive gap = FEI recognized better than Celeb-DF++):")
    display(piv.round(4))

# Bar chart for the thesis figure: one panel per metric, FEI vs Celeb-DF++ bars per model
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
models_list = list(trained_models.keys())
x = np.arange(len(models_list))
width = 0.35

for ax, metric in zip(axes, metrics):
    fei_vals = results_fei.set_index("Model").loc[models_list, metric]
    celeb_vals = results_celeb.set_index("Model").loc[models_list, metric]

    ax.bar(x - width / 2, fei_vals, width, label="FEI", color="#3b6ea5")
    ax.bar(x + width / 2, celeb_vals, width, label="Celeb-DF++", color="#c97a3d")
    ax.set_xticks(x)
    ax.set_xticklabels(models_list, rotation=30, ha="right", fontsize=8)
    ax.set_title(metric)
    ax.legend(fontsize=8)

axes[-1].axis("off")
fig.suptitle("Test metrics by source dataset (FEI vs Celeb-DF++)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("fei_vs_celeb_all_metrics.png", dpi=200)
plt.show()